# **Installation and Licensing**

In [ ]:
!pip install -q gamspy # Installs the GAMSPy package. GAMSPy documentation: https://gamspy.readthedocs.io/en/latest/

In [ ]:
!gamspy show license

In [ ]:
!gamspy list solvers

In [ ]:
!gamspy install license <path_to_ascii_file or access code> # paste your license here (if demo-license is not enough)

In [ ]:
# Optional: Install all solvers
!gamspy install solver --install-all-solvers

In [ ]:
!gamspy list solvers # Lists all available solvers in the gamspy package

# **Mathematical Model**

**Cardinality-Constrained Portfolio Optimization with Risk, Tracking Error, Diversification, and Transaction Costs**

\begin{align}
    \min \quad & \sum_{i=1}^n \sum_{j=1}^n Q_{ij} x_i x_j - \sum_{i=1}^n \mu_i x_i + \sum_{i=1}^n c_i (x_i - x_i^{prev})^2 \\
    \text{subject to} \quad
    & \sum_{i=1}^n x_i = 1 \quad && \text{(Budget)} \\
    & x_i \leq y_i, \quad \forall i \quad && \text{(Linking)} \\
    & \sum_{i=1}^n y_i \leq 5 \quad && \text{(Cardinality)} \\
    & \sum_{i=1}^n \sum_{j=1}^n Q_{ij} x_i x_j \leq \tau = 0.05 \quad && \text{(Risk)} \\
    & \sum_{i=1}^n \sum_{j=1}^n Q_{ij} (x_i - x_i^{bench})(x_j - x_j^{bench}) \leq \delta = 0.001 \quad && \text{(Tracking Error)} \\
    & \sum_{i=1}^n \sum_{j=1}^n M_{ij} x_i x_j \leq 1 \quad && \text{(Diversification)} \\
    & x_i \in [0,1], \quad y_i \in \{0,1\}, \quad \forall i \quad && \text{(Domains)}
\end{align}

$$
\begin{array}{l l l r r r r}
\text{Name} & \text{Type} & \text{C} & \#\text{Vars} & \#\text{BinVars} & \#\text{IntVars} & \#\text{Cons} \\
\hline
CCPOPwTCRTEaD5\_20 & MBQCQP & Convex & 40 & 20 & 0 & 25 \\
CCPOPwTCRTEaD5\_50 & MBQCQP & Convex & 100 & 50	& 0 & 55 \\
\end{array}
$$


#  **Solver Setup**


## CCPOPwTCRTEaD5\_20

In [ ]:
import yfinance as yf
import numpy as np
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# --- 1. Data ---
tickers = [
    "PSA", "EMR", "ETN", "ROP", "AIG",
    "MAR", "USB", "D", "EW", "IDXX", "AEP", "FTNT", "ADSK",
    "ROST", "MNST", "MET", "CTAS", "AZO",
    "SPG", "WTW"
]
data = yf.download(tickers, start="2023-01-01", end="2024-05-31")["Close"]
returns = data.pct_change().dropna()

mu_array = returns.mean().values
cov_matrix = returns.cov().values
n_assets = len(mu_array)
assets = [f"asset_{i+1}" for i in range(n_assets)]

# --- 2. Model container and sets ---
m = Container()
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

# --- 3. Parameters ---
Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov_matrix[r, c])
                       for r in range(n_assets) for c in range(n_assets)])
mu = Parameter(m, name="mu", domain=[i],
               records=[(assets[k], mu_array[k]) for k in range(n_assets)])
c_array = np.random.uniform(0.001, 0.01, size=n_assets)  # transaction cost coefficients

# Previous portfolio (for transaction cost)
x_prev_array = np.random.rand(n_assets)
x_prev_array = x_prev_array / x_prev_array.sum()  # normalize to sum to 1

x_prev = Parameter(m, name="x_prev", domain=[i],
                   records=[(assets[k], x_prev_array[k]) for k in range(n_assets)])

c = Parameter(m, name="c", domain=[i],
              records=[(assets[k], c_array[k]) for k in range(n_assets)])


# Benchmark portfolio for tracking error (optional)
x_bench_array = np.array([0.05] * n_assets)  # equal weight 5%
x_bench = Parameter(m, name="x_bench", domain=[i],
                    records=[(assets[k], x_bench_array[k]) for k in range(n_assets)])

# --- 4. Model parameters ---
K = 5       # max assets to select
tau = 0.05 # max portfolio variance
delta = 0.001 # max tracking error

# --- 5. Variables ---
x = Variable(m, name="x", domain=[i], type="Free")
x.lo[i] = 0
x.up[i] = 1
y = Variable(m, name="y", domain=[i], type="Binary")

# --- 6. Constraints ---

# Budget: Invest 100%
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

# Linking x and y: x_i <= y_i
link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

# Cardinality: sum y_i <= K
card = Equation(m, name="cardinality")
card[...] = Sum(i, y[i]) <= K

# Risk: x^T Q x <= tau
risk = Equation(m, name="risk")
risk[...] = Sum([i, ii], Q[i, ii] * x[i] * x[ii]) <= tau

# Tracking error: (x - x_bench)^T Q (x - x_bench) <= delta
tracking = Equation(m, name="tracking")
tracking[...] = Sum([i, ii], Q[i, ii] * (x[i] - x_bench[i]) * (x[ii] - x_bench[ii])) <= delta


# Define custom matrix M (e.g., random symmetric PSD for now)
random_matrix = np.random.randn(n_assets, n_assets)
M_matrix = np.dot(random_matrix.T, random_matrix)  # Make it symmetric PSD
M_matrix_normalized = M_matrix / M_matrix.max() # Normalize

M = Parameter(m, name="M", domain=[i, ii],
              records=[(assets[r], assets[c], M_matrix_normalized[r, c])
                       for r in range(n_assets) for c in range(n_assets)])

# Add quadratic constraint: x^T M x <= diversification budget
diversify = Equation(m, name="diversify")
diversify[...] = Sum([i, ii], M[i, ii] * x[i] * x[ii]) <= 1  # or any reasonable value

# --- 7. Objective ---
risk_term = Sum([i, ii], Q[i, ii] * x[i] * x[ii])
ret_term = Sum(i, mu[i] * x[i])
trans_cost_term = Sum(i, c[i] * (x[i] - x_prev[i]) * (x[i] - x_prev[i]))


obj = risk_term - ret_term + trans_cost_term

# --- 8. Build and solve model ---
model = Model(m, name="CCPOPwTCRTEaD_5_20",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MIN,
              objective=obj)

import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})


In [ ]:
# Convert GAMSPy model to GAMS (Note: One needs to download both the .gms and .gdx file)
model.toGams(path="gams")

After this, the files were opened and combined in GAMS Studio by GAMS Convert. Below is the rest of the codes for this type of problem.


## CCPOPwTCRTEaD5\_50

In [ ]:
import yfinance as yf
import numpy as np
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# --- 1. Data ---
tickers = [
    "MMC", "ADP", "VRTX", "CB", "ELV", "REGN", "PNC", "FI", "C", "ADI",
    "CSX", "MDLZ", "SCHW", "PGR", "MU", "DUK", "EQIX", "SO", "HCA",
    "NKE", "PSA", "EMR", "ETN", "ROP", "AIG",
    "MAR", "USB", "D", "EW", "IDXX", "AEP", "FTNT", "ADSK",
    "ROST", "MNST", "MET", "CTAS", "AZO",
    "SPG", "WTW", "CME", "DLR", "PRU", "MSI",
    "ALL", "WELL", "TEL", "SRE", "HLT", "HSY"
]
data = yf.download(tickers, start="2023-01-01", end="2024-05-31")["Close"]
returns = data.pct_change().dropna()

mu_array = returns.mean().values
cov_matrix = returns.cov().values
n_assets = len(mu_array)
assets = [f"asset_{i+1}" for i in range(n_assets)]

# --- 2. Model container and sets ---
m = Container()
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

# --- 3. Parameters ---
Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov_matrix[r, c])
                       for r in range(n_assets) for c in range(n_assets)])
mu = Parameter(m, name="mu", domain=[i],
               records=[(assets[k], mu_array[k]) for k in range(n_assets)])
c_array = np.random.uniform(0.001, 0.01, size=n_assets)  # transaction cost coefficients

# Previous portfolio (for transaction cost)
x_prev_array = np.random.rand(n_assets)
x_prev_array = x_prev_array / x_prev_array.sum()  # normalize to sum to 1

x_prev = Parameter(m, name="x_prev", domain=[i],
                   records=[(assets[k], x_prev_array[k]) for k in range(n_assets)])

c = Parameter(m, name="c", domain=[i],
              records=[(assets[k], c_array[k]) for k in range(n_assets)])


# Benchmark portfolio for tracking error (optional)
x_bench_array = np.array([0.05] * n_assets)  # equal weight 5%
x_bench = Parameter(m, name="x_bench", domain=[i],
                    records=[(assets[k], x_bench_array[k]) for k in range(n_assets)])

# --- 4. Model parameters ---
K = 5       # max assets to select
tau = 0.05 # max portfolio variance
delta = 0.001 # max tracking error

# --- 5. Variables ---
x = Variable(m, name="x", domain=[i], type="Free")
x.lo[i] = 0
x.up[i] = 1
y = Variable(m, name="y", domain=[i], type="Binary")

# --- 6. Constraints ---

# Budget: Invest 100%
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

# Linking x and y: x_i <= y_i
link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

# Cardinality: sum y_i <= K
card = Equation(m, name="cardinality")
card[...] = Sum(i, y[i]) <= K

# Risk: x^T Q x <= tau
risk = Equation(m, name="risk")
risk[...] = Sum([i, ii], Q[i, ii] * x[i] * x[ii]) <= tau

# Tracking error: (x - x_bench)^T Q (x - x_bench) <= delta
tracking = Equation(m, name="tracking")
tracking[...] = Sum([i, ii], Q[i, ii] * (x[i] - x_bench[i]) * (x[ii] - x_bench[ii])) <= delta


# Define custom matrix M (e.g., random symmetric PSD for now)
random_matrix = np.random.randn(n_assets, n_assets)
M_matrix = np.dot(random_matrix.T, random_matrix)  # Make it symmetric PSD
M_matrix_normalized = M_matrix / M_matrix.max() # Normalize

M = Parameter(m, name="M", domain=[i, ii],
              records=[(assets[r], assets[c], M_matrix_normalized[r, c])
                       for r in range(n_assets) for c in range(n_assets)])

# Add quadratic constraint: x^T M x <= diversification budget
diversify = Equation(m, name="diversify")
diversify[...] = Sum([i, ii], M[i, ii] * x[i] * x[ii]) <= 1  # or any reasonable value

# --- 7. Objective ---
risk_term = Sum([i, ii], Q[i, ii] * x[i] * x[ii])
ret_term = Sum(i, mu[i] * x[i])
trans_cost_term = Sum(i, c[i] * (x[i] - x_prev[i]) * (x[i] - x_prev[i]))


obj = risk_term - ret_term + trans_cost_term

# --- 8. Build and solve model ---
model = Model(m, name="CCPOPwTCRTEaD_5_50",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MIN,
              objective=obj)

import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})
